# Defter 1 (Sürüm 2, son çalıştırma) — Veri tensörleri (ITRDB + CRU TS4.09 + yetişme ortamı)
**Embedding tree growth theory into AI** · Prof. Dr. İlker Ercanlı

Bu defter bir kez çalıştırılır. Yaptıkları:
1. Halka ve iklim tablolarını Google Drive'dan okur.
2. VS-Lite kova modeliyle aylık toprak nemini (M) ve normalleştirilmiş gün uzunluğunu (g_E) hesaplar.
3. Dört test bölünmesini kurar (zamansal, kıta, 10° blok, taksonomik).
4. Yetişme ortamı değişkenlerini (toprak, topoğrafya, biyom, ekolojik bölge, Köppen) ekler; koordinatı hatalı alanları çıkarır.
5. Her şeyi `veri_v2.npz` dosyasına yazar. Defter 2 (son) bu dosyayı okur.

**Ek gereksinim:** `ITRDB_AI/alan_ortam.csv` (D:\ITRDB_envanter\ORTAM\alan_ortam.csv dosyasının kopyası).

**Önce:** Drive'da `ITRDB_AI` klasörü olmalı ve içinde `HAZIR/` klasörü ile `iklim_aylik.csv.gz` bulunmalı.
GPU gerekmez; bu defter CPU'da 5–10 dakika sürer.

In [ ]:
# --- Drive bağlantısı ve klasörler ---
import os, json, time, numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ANA = '/content/drive/MyDrive/ITRDB_AI'
except ImportError:                       # Colab dışında (yerel deneme) çalışırsa
    ANA = os.environ.get('ITRDB_AI', '.')
HAZIR = os.path.join(ANA, 'HAZIR')
IKLIM = os.path.join(ANA, 'iklim_aylik.csv.gz')
ORTAM = os.path.join(ANA, 'alan_ortam.csv')
CIKTI = os.path.join(ANA, 'TENSOR'); os.makedirs(CIKTI, exist_ok=True)
for yol in [ORTAM, HAZIR, IKLIM, os.path.join(HAZIR, 'halka'), os.path.join(HAZIR, 'seri_meta.csv')]:
    assert os.path.exists(yol), f'Bulunamadı: {yol}  -> Drive klasörünü kontrol edin'
print('Klasörler tamam:', ANA)

YIL0, YIL1 = 1901, 2024
NY = YIL1 - YIL0 + 1                      # 124 yıl
TAKSON_TEST = ['Larix', 'Fagus', 'Cedrus', 'Nothofagus']
ZAMAN_ESIK = 30                           # 1960'ın iki yanında en az bu kadar halka
TOHUM = 2026
t0 = time.time()

In [ ]:
# --- 1. Seri ve alan bilgileri ---
seri = pd.read_csv(os.path.join(HAZIR, 'seri_meta.csv'))
alan = pd.read_csv(os.path.join(HAZIR, 'alan_meta.csv'))
alan = alan[alan.studyCode.isin(seri.studyCode)].sort_values('studyCode').reset_index(drop=True)
# Veri düzeltmesi (analiz planı 1a): kıta etiketi ile koordinatı çelişen alanlar (K. Amerika etiketli, boylam > -50)
hatali = alan[(alan.continent == 'North America') & (alan.lon > -50)].studyCode.tolist()
print('Koordinatı hatalı, çıkarılan alanlar:', hatali)
alan = alan[~alan.studyCode.isin(hatali)].reset_index(drop=True)
seri = seri[seri.studyCode.isin(alan.studyCode)]
alan['site_idx'] = np.arange(len(alan))
seri = seri.merge(alan[['studyCode', 'site_idx']], on='studyCode').sort_values(['site_idx', 'seri_uid']).reset_index(drop=True)
seri['series_idx'] = np.arange(len(seri))
seri['tree_idx'] = pd.factorize(seri.agac_kodu_tahmini)[0]
alan['genus'] = alan.genus.fillna('Unknown'); alan['scientificName'] = alan.scientificName.fillna('Unknown')
genus_ad = sorted(alan.genus.unique()); tur_ad = sorted(alan.scientificName.unique())
alan['genus_idx'] = alan.genus.map({g: i for i, g in enumerate(genus_ad)})
alan['species_idx'] = alan.scientificName.map({s: i for i, s in enumerate(tur_ad)})
print(f'{len(alan)} alan, {len(seri)} seri, {seri.tree_idx.max()+1} ağaç, {len(tur_ad)} tür, {len(genus_ad)} cins')

In [ ]:
# --- 2. Halkalar ---
import glob
parca = sorted(glob.glob(os.path.join(HAZIR, 'halka', 'part_*.parquet')))
H = pd.concat([pd.read_parquet(p) for p in parca], ignore_index=True)
H = H.merge(seri[['seri_uid', 'series_idx']], on='seri_uid', how='inner')
H = H[(H.year >= YIL0) & (H.year <= YIL1)].sort_values(['series_idx', 'year']).reset_index(drop=True)
ring_series = H.series_idx.to_numpy(np.int32)
ring_year = H.year.to_numpy(np.int16)
ring_rw = H.rw_mm.to_numpy(np.float32)
ring_rprev = H.r_onceki_mm.to_numpy(np.float32)
del H
print(f'{len(ring_rw):,} halka; sıfır halka oranı {np.mean(ring_rw == 0):.4f}; medyan genişlik {np.median(ring_rw):.2f} mm')

In [ ]:
# --- 3. İklim: [alan, yıl, ay, değişken] ---
DEG = ['tmp', 'pre', 'pet', 'vpd']
ik = pd.read_csv(IKLIM, usecols=['studyCode', 'year', 'month'] + DEG + ['stn_tmp', 'stn_pre'])
ik = ik.merge(alan[['studyCode', 'site_idx']], on='studyCode', how='inner')
S = len(alan)
clim = np.full((S, NY, 12, len(DEG)), np.nan, np.float32)
stn = np.zeros((S, NY, 12, 2), np.int16)
iy, im, isx = ik.year.to_numpy() - YIL0, ik.month.to_numpy() - 1, ik.site_idx.to_numpy()
for k, v in enumerate(DEG): clim[isx, iy, im, k] = ik[v].to_numpy()
stn[isx, iy, im, 0] = ik.stn_tmp.fillna(0).to_numpy(); stn[isx, iy, im, 1] = ik.stn_pre.fillna(0).to_numpy()
del ik
eksik = np.isnan(clim[..., 0]).all(axis=(1, 2))
print('İklimi eksik alan:', int(eksik.sum()))
assert eksik.sum() == 0, 'Bazı alanların iklimi yok; alan_meta ile iklim tablosunu kontrol edin'

In [ ]:
# --- 4. VS-Lite: gün uzunluğu ve kova modeliyle toprak nemi ---
# Tolwinski-Ward ve ark. (2011, 2013) VS-Lite; Huang ve ark. (1996) kova modeli. Parametreler VS-Lite varsayılanları.
def gun_uzunlugu(lat):
    """Ay ortası gün uzunluğu (saat): [alan, 12]"""
    gun = np.array([15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349])
    dek = 23.45 * np.sin(np.deg2rad(360 * (284 + gun) / 365))
    x = -np.tan(np.deg2rad(lat))[:, None] * np.tan(np.deg2rad(dek))[None, :]
    return 24 / np.pi * np.arccos(np.clip(x, -1, 1))

def kova_modeli(T, P, lat, Mmax=0.76, Mmin=0.01, alph=0.093, m_th=4.886, mu_th=5.80, rootd=1000., M0=0.2):
    """T, P: [alan, ay]; dönüş M (v/v): [alan, ay]"""
    S, N = T.shape
    gu = gun_uzunlugu(lat)                                 # [S,12]
    ngun = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
    L = (ngun / 30)[None, :] * (gu / 12)                   # Thornthwaite düzeltmesi
    Tclim = np.nanmean(T.reshape(S, -1, 12), axis=1)       # alanın aylık normali
    I = np.sum(np.where(Tclim > 0, (np.maximum(Tclim, 0) / 5) ** 1.514, 0), axis=1)
    a = 6.75e-7 * I**3 - 7.71e-5 * I**2 + 1.792e-2 * I + 0.49
    M = np.empty((S, N), np.float32); m = np.full(S, M0)
    for t in range(N):
        Tt, Pt, Lt = T[:, t], P[:, t], L[:, t % 12]
        Ep = np.where(Tt <= 0, 0., np.where(Tt < 26.5, 16 * Lt * (10 * np.maximum(Tt, 0) / np.maximum(I, 1e-6)) ** a,
                                            -415.85 + 32.25 * Tt - 0.43 * Tt**2))
        Ep = np.maximum(Ep, 0)
        E = Ep * (m / Mmax)
        G = mu_th * alph / (1 + mu_th) * m * rootd
        R = Pt * (m / Mmax) ** m_th + (alph / (1 + mu_th)) * m * rootd
        m = np.clip(m + (Pt - E - R - G) / rootd, Mmin, Mmax)
        M[:, t] = m
    return M, gu

T = clim[..., 0].reshape(S, -1); P = clim[..., 1].reshape(S, -1)
M, gu = kova_modeli(T, P, alan.lat.to_numpy())
gE = (gu / gu.max(axis=1, keepdims=True)).astype(np.float32)   # VS-Lite g_E: [alan, 12]
soil = M.reshape(S, NY, 12)
print(f'Toprak nemi: ortalama {soil.mean():.3f} v/v, aralık {soil.min():.2f}–{soil.max():.2f}')

In [ ]:
# --- 5. Test bölünmeleri ---
rng = np.random.default_rng(TOHUM)
# (a) Zamansal: 1960'ın iki yanında en az 30 halka olan seriler; eğitim ≤1960, test >1960
seri['zaman_uygun'] = (seri.n_once_1960 >= ZAMAN_ESIK) & (seri.n_sonra_1960 >= ZAMAN_ESIK)
# (b) Kıta: dört bölge sırayla dışarıda bırakılır; Güney Yarımküre enlem < 0 ile tanımlanır
# Bölgeler koordinata göre (analiz planı 3): Sibirya Asya'da, Anadolu Avrupa'da
bolge = np.full(len(alan), 'Diger', dtype=object)
bolge[(alan.lon < -30) & (alan.lat >= 0)] = 'North America'
bolge[(alan.lon >= -30) & (alan.lon < 60) & (alan.lat >= 34)] = 'Europe'
bolge[(alan.lon >= 60) & (alan.lat >= 0)] = 'Asia'
bolge[alan.lat < 0] = 'Southern Hemisphere'
alan['bolge'] = bolge
# (c) 10° bloklar 5 kata rastgele dağıtılır
bloklar = alan.blok_10.unique(); kat = dict(zip(bloklar, rng.permutation(len(bloklar)) % 5))
alan['blok_kat'] = alan.blok_10.map(kat)
# (d) Taksonomik
alan['takson_test'] = alan.genus.isin(TAKSON_TEST)
# İç doğrulama (erken durdurma): alanların %10'u
alan['ic_dogrulama'] = rng.random(len(alan)) < 0.10
print(alan.bolge.value_counts().to_string()); print('Taksonomik test alanı:', int(alan.takson_test.sum()))
print('Zamansal teste uygun seri:', int(seri.zaman_uygun.sum()), '/ alan:', seri[seri.zaman_uygun].studyCode.nunique())

In [ ]:
# --- 6. Alan tanımlayıcıları (parametre ağının girdisi z_s) ---
nrm = slice(0, 60)                                         # 1901–1960
T_n = np.nanmean(clim[:, nrm, :, 0], axis=(1, 2)); P_n = np.nansum(clim[:, nrm, :, 1], axis=2).mean(1)
PET_n = np.nansum(clim[:, nrm, :, 2], axis=2).mean(1); VPD_n = np.nanmean(clim[:, nrm, :, 3], axis=(1, 2))
M_n = soil[:, nrm].mean(axis=(1, 2))
# --- Yetişme ortamı değişkenleri ---
ort = pd.read_csv(ORTAM)
alan = alan.merge(ort.drop(columns=['lat', 'lon', 'elev_m']), on='studyCode', how='left')
ENV = ['egim_derece', 'kuzeylik', 'tpi_m', 'toprak_sand', 'toprak_clay', 'toprak_soc', 'toprak_phh2o',
       'toprak_bdod', 'toprak_cfvo', 'toprak_nitrogen', 'toprak_faydali_su']
alan['ortam_dolduruldu'] = alan[ENV].isna().any(axis=1)
for v in ENV:   # eksikler: ekolojik bölge ortancası -> biyom ortancası -> genel ortanca (analiz planı 1b)
    alan[v] = alan[v].fillna(alan.groupby('eko_id')[v].transform('median')).fillna(alan.groupby('biyom_no')[v].transform('median')).fillna(alan[v].median())
env = alan[ENV].copy()
for v in ['toprak_soc', 'toprak_nitrogen', 'tpi_m']:
    env[v] = np.sign(env[v]) * np.log1p(np.abs(env[v]))           # çarpık dağılımlar
env = env.to_numpy(np.float32)
def kod(x):
    x = x.fillna(-1).astype(int); u = sorted(set(x) - {-1})
    return x.map({k: i for i, k in enumerate(u)}).fillna(len(u)).astype(int).to_numpy(), len(u)   # son indeks = bilinmeyen
site_biyom, n_biyom = kod(alan.biyom_no); site_eko, n_eko = kod(alan.eko_id); site_kg, n_kg = kod(alan.koppen_no)
alan['yukselti_bayrak'] = (alan.elev_m - alan.dem_yukselti).abs() > 500   # analiz planı 1, kalite bayrağı
print(f'Ortam: {len(ENV)} sürekli değişken; biyom {n_biyom}, ekolojik bölge {n_eko}, Köppen {n_kg} sınıf; '
      f'doldurulan alan {int(alan.ortam_dolduruldu.sum())}; yükselti bayraklı alan {int(alan.yukselti_bayrak.sum())}')
assert np.isfinite(env).all()
elev = alan.elev_m.fillna(alan.dem_yukselti).fillna(alan.elev_m.median()).to_numpy()
desc = np.stack([alan.lat, elev / 1000, T_n, np.log1p(P_n), np.log1p(PET_n), P_n / np.maximum(PET_n, 1), VPD_n, M_n], 1).astype(np.float32)
desc_ad = ['lat', 'elev_km', 'T_norm', 'logP_norm', 'logPET_norm', 'aridity', 'VPD_norm', 'M_norm']
assert np.isfinite(desc).all()

In [ ]:
# --- 7. Kaydet ---
np.savez_compressed(os.path.join(CIKTI, 'veri_v2.npz'),
    env=env, site_biyom=site_biyom, site_eko=site_eko, site_kg=site_kg,
    site_yukselti_bayrak=alan.yukselti_bayrak.to_numpy(bool), site_ortam_dolduruldu=alan.ortam_dolduruldu.to_numpy(bool),
    ring_series=ring_series, ring_year=ring_year, ring_rw=ring_rw, ring_rprev=ring_rprev,
    series_site=seri.site_idx.to_numpy(np.int32), series_tree=seri.tree_idx.to_numpy(np.int32),
    series_zaman=seri.zaman_uygun.to_numpy(bool), r_cru_basi=seri.r_cru_basi_mm.to_numpy(np.float32),
    clim=clim, soil=soil, gE=gE, stn=stn, desc=desc,
    site_genus=alan.genus_idx.to_numpy(np.int32), site_species=alan.species_idx.to_numpy(np.int32),
    site_lat=alan.lat.to_numpy(np.float32), site_lon=alan.lon.to_numpy(np.float32),
    site_blok_kat=alan.blok_kat.to_numpy(np.int8), site_takson_test=alan.takson_test.to_numpy(bool),
    site_ic_dogrulama=alan.ic_dogrulama.to_numpy(bool))
json.dump({'genus': genus_ad, 'species': tur_ad, 'clim_vars': DEG, 'desc_vars': desc_ad,
           'site_code': alan.studyCode.tolist(), 'site_bolge': alan.bolge.tolist(), 'env_vars': ENV, 'n_biyom': n_biyom, 'n_eko': n_eko, 'n_kg': n_kg, 'cikarilan': hatali,
           'series_uid': seri.seri_uid.tolist(), 'yil0': YIL0, 'takson_test': TAKSON_TEST, 'tohum': TOHUM},
          open(os.path.join(CIKTI, 'veri_v2_sozluk.json'), 'w'))
alan.to_csv(os.path.join(CIKTI, 'alan_bolunme_v2.csv'), index=False)
boy = os.path.getsize(os.path.join(CIKTI, 'veri_v2.npz')) / 1e6
print(f'\n==== TAMAM ====  veri_v2.npz {boy:.0f} MB, süre {(time.time()-t0)/60:.1f} dk')